# TS-ResNet50 Teacher-Layer Ablation Notebook

This notebook is the runnable home for the `TS-ResNet50` teacher-layer comparison.

It is designed to work in two modes:
- analysis mode: reuse imported Kaggle artifacts and saved leaderboard snapshots that already exist in the repo
- run mode: generate local configs, train selected `TS-ResNet50` variants from this repo, evaluate them, and save a local summary table

The full sweep is GPU-heavy, which is why some earlier runs were executed on Kaggle. Even so, the notebook is runnable from this repo for anyone who wants to reproduce or extend the experiment locally.

In [1]:
from pathlib import Path
from copy import deepcopy
import json
import subprocess
import sys

from IPython.display import display
import pandas as pd
import torch

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml

REPO_ROOT

WindowsPath('C:/Users/User/Desktop/Term 8/Deep Learning/Project/DeepLearning-Group8')

In [2]:
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "training" / "train_ts_resnet50_kaggle.toml"
IMPORTED_LAYER2_DIR = REPO_ROOT / "artifacts" / "x64" / "ts_resnet50"
KAGGLE_ABLATION_DIR = REPO_ROOT / "artifacts" / "x64" / "kaggle_rn50_ablation"
GENERATED_CONFIG_DIR = REPO_ROOT / "artifacts" / "generated_configs"
LOCAL_SUMMARY_PATH = REPO_ROOT / "artifacts" / "x64" / "ts_resnet50_teacher_layer_ablation_summary.csv"

RUN_VARIANT_SWEEP = False
REUSE_EXISTING_LOCAL_RUNS = True
LOAD_IMPORTED_LAYER2_RESULTS = True
LOAD_KAGGLE_LAYER1_LEADERBOARDS = True
THRESHOLD_QUANTILE = 0.95

VARIANT_SPECS = [
    {
        "name": "ts_resnet50_layer2_topk20_sw2p0_aw1p0",
        "teacher_layer": "layer2",
        "topk_ratio": 0.20,
        "score_student_weight": 2.0,
        "score_autoencoder_weight": 1.0,
    },
    {
        "name": "ts_resnet50_layer1_topk10_sw1p0_aw0p5",
        "teacher_layer": "layer1",
        "topk_ratio": 0.10,
        "score_student_weight": 1.0,
        "score_autoencoder_weight": 0.5,
    },
    {
        "name": "ts_resnet50_layer1_topk10_sw2p0_aw1p0",
        "teacher_layer": "layer1",
        "topk_ratio": 0.10,
        "score_student_weight": 2.0,
        "score_autoencoder_weight": 1.0,
    },
    {
        "name": "ts_resnet50_layer1_topk10_sw2p0_aw0p5",
        "teacher_layer": "layer1",
        "topk_ratio": 0.10,
        "score_student_weight": 2.0,
        "score_autoencoder_weight": 0.5,
    },
    {
        "name": "ts_resnet50_layer1_topk20_sw2p0_aw1p0",
        "teacher_layer": "layer1",
        "topk_ratio": 0.20,
        "score_student_weight": 2.0,
        "score_autoencoder_weight": 1.0,
    },
]

base_config = load_toml(BASE_CONFIG_PATH)
resolved_device_name = "cuda" if torch.cuda.is_available() else "cpu"
display(base_config)
print(f"Base config: {BASE_CONFIG_PATH.name}")
print(f"Resolved device for local runs: {resolved_device_name}")
print(f"RUN_VARIANT_SWEEP={RUN_VARIANT_SWEEP}")

{'run': {'output_dir': 'artifacts/x64/ts_resnet50', 'seed': 42},
 'data': {'metadata_csv': 'data/processed/x64/wm811k/metadata_50k_5pct.csv',
  'image_size': 64,
  'batch_size': 16,
  'num_workers': 0},
 'training': {'epochs': 30,
  'learning_rate': 0.0003,
  'weight_decay': 1e-05,
  'device': 'auto',
  'early_stopping_patience': 5,
  'early_stopping_min_delta': 0.0001,
  'checkpoint_every': 5,
  'resume_from': ''},
 'model': {'type': 'ts_distillation',
  'teacher_backbone': 'resnet50',
  'teacher_layer': 'layer2',
  'teacher_pretrained': True,
  'teacher_input_size': 224,
  'normalize_teacher_input': True,
  'feature_autoencoder_hidden_dim': 128,
  'student_weight': 1.0,
  'autoencoder_weight': 1.0,
  'score_student_weight': 1.0,
  'score_autoencoder_weight': 0.0,
  'reduction': 'topk_mean',
  'topk_ratio': 0.2}}

Base config: train_ts_resnet50_kaggle.toml
Resolved device for local runs: cuda
RUN_VARIANT_SWEEP=False


In [3]:
def format_toml_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return repr(value)
    return json.dumps(str(value))


def write_simple_toml(path, config_dict):
    lines = []
    for section_name, section_values in config_dict.items():
        lines.append(f"[{section_name}]")
        for key, value in section_values.items():
            lines.append(f"{key} = {format_toml_value(value)}")
        lines.append("")
    path.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")


def stream_command(command, cwd):
    print("Running:", " ".join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def build_variant_config(base_config, spec):
    config = deepcopy(base_config)
    config["run"]["output_dir"] = f"artifacts/x64/{spec['name']}"
    config["model"]["teacher_layer"] = spec["teacher_layer"]
    config["model"]["topk_ratio"] = float(spec["topk_ratio"])
    config["model"]["score_student_weight"] = float(spec["score_student_weight"])
    config["model"]["score_autoencoder_weight"] = float(spec["score_autoencoder_weight"])
    config["training"]["resume_from"] = ""
    return config


def run_variant(spec, threshold_quantile, reuse_existing=True):
    variant_config = build_variant_config(base_config, spec)
    GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    config_path = GENERATED_CONFIG_DIR / f"{spec['name']}.toml"
    write_simple_toml(config_path, variant_config)

    output_dir = REPO_ROOT / variant_config["run"]["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / "best_model.pt"
    evaluation_dir = output_dir / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    summary_path = evaluation_dir / "summary.json"

    if reuse_existing and checkpoint_path.exists():
        print(f"Reusing checkpoint for {spec['name']}: {checkpoint_path}")
    else:
        stream_command(
            [
                sys.executable,
                "-u",
                REPO_ROOT / "scripts" / "train_ts_distillation.py",
                "--config",
                config_path,
            ],
            cwd=REPO_ROOT,
        )

    if reuse_existing and summary_path.exists():
        print(f"Reusing evaluation for {spec['name']}: {summary_path}")
    else:
        stream_command(
            [
                sys.executable,
                REPO_ROOT / "scripts" / "evaluate_reconstruction_model.py",
                "--checkpoint",
                checkpoint_path,
                "--config",
                config_path,
                "--model-type",
                "ts_distillation",
                "--threshold-quantile",
                str(threshold_quantile),
                "--output-dir",
                evaluation_dir,
            ],
            cwd=REPO_ROOT,
        )

    train_summary = json.loads((output_dir / "summary.json").read_text(encoding="utf-8"))
    evaluation_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    metrics = evaluation_summary["metrics_at_validation_threshold"]
    best_sweep = evaluation_summary["best_threshold_sweep"]

    return {
        "variant": spec["name"],
        "source": "local_run",
        "teacher_layer": spec["teacher_layer"],
        "topk_ratio": float(spec["topk_ratio"]),
        "score_student_weight": float(spec["score_student_weight"]),
        "score_autoencoder_weight": float(spec["score_autoencoder_weight"]),
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1": float(metrics["f1"]),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "best_sweep_f1": float(best_sweep["f1"]),
        "threshold": float(metrics["threshold"]),
        "best_epoch": int(train_summary["best_epoch"]),
        "best_val_loss": float(train_summary["best_val_loss"]),
        "output_dir": str(output_dir),
        "config_path": str(config_path),
    }


pd.DataFrame(VARIANT_SPECS)

,name,teacher_layer,topk_ratio,score_student_weight,score_autoencoder_weight
0,ts_resnet50_layer2_topk20_sw2p0_aw1p0,layer2,0.2,2.0,1.0
1,ts_resnet50_layer1_topk10_sw1p0_aw0p5,layer1,0.1,1.0,0.5
2,ts_resnet50_layer1_topk10_sw2p0_aw1p0,layer1,0.1,2.0,1.0
3,ts_resnet50_layer1_topk10_sw2p0_aw0p5,layer1,0.1,2.0,0.5
4,ts_resnet50_layer1_topk20_sw2p0_aw1p0,layer1,0.2,2.0,1.0


In [4]:
analysis_rows = []

if LOAD_IMPORTED_LAYER2_RESULTS:
    imported_summary_path = IMPORTED_LAYER2_DIR / "evaluation" / "summary.json"
    selected_summary_path = IMPORTED_LAYER2_DIR / "evaluation_local" / "summary.json"
    if imported_summary_path.exists() and selected_summary_path.exists():
        imported_summary = json.loads(imported_summary_path.read_text(encoding="utf-8"))
        selected_summary = json.loads(selected_summary_path.read_text(encoding="utf-8"))
        analysis_rows.extend([
            {
                "variant": "TS-Res50 imported default",
                "source": "imported_layer2",
                "teacher_layer": "layer2",
                "topk_ratio": 0.20,
                "score_student_weight": 1.0,
                "score_autoencoder_weight": 0.0,
                "precision": imported_summary["metrics_at_validation_threshold"]["precision"],
                "recall": imported_summary["metrics_at_validation_threshold"]["recall"],
                "f1": imported_summary["metrics_at_validation_threshold"]["f1"],
                "auroc": imported_summary["metrics_at_validation_threshold"]["auroc"],
                "auprc": imported_summary["metrics_at_validation_threshold"]["auprc"],
                "best_sweep_f1": imported_summary["best_threshold_sweep"]["f1"],
                "threshold": imported_summary["metrics_at_validation_threshold"]["threshold"],
                "best_epoch": imported_summary.get("best_epoch", None),
                "best_val_loss": imported_summary.get("best_val_loss", None),
                "output_dir": str(IMPORTED_LAYER2_DIR),
                "config_path": str(BASE_CONFIG_PATH),
            },
            {
                "variant": "TS-Res50 selected local score",
                "source": "imported_layer2",
                "teacher_layer": "layer2",
                "topk_ratio": 0.20,
                "score_student_weight": 2.0,
                "score_autoencoder_weight": 1.0,
                "precision": selected_summary["metrics_at_validation_threshold"]["precision"],
                "recall": selected_summary["metrics_at_validation_threshold"]["recall"],
                "f1": selected_summary["metrics_at_validation_threshold"]["f1"],
                "auroc": selected_summary["metrics_at_validation_threshold"]["auroc"],
                "auprc": selected_summary["metrics_at_validation_threshold"]["auprc"],
                "best_sweep_f1": selected_summary["best_threshold_sweep"]["f1"],
                "threshold": selected_summary["metrics_at_validation_threshold"]["threshold"],
                "best_epoch": selected_summary.get("best_epoch", None),
                "best_val_loss": selected_summary.get("best_val_loss", None),
                "output_dir": str(IMPORTED_LAYER2_DIR),
                "config_path": str(BASE_CONFIG_PATH),
            },
        ])
    else:
        print("Imported layer2 summaries were not found. Skipping imported-layer2 rows.")

if LOAD_KAGGLE_LAYER1_LEADERBOARDS:
    leaderboard_paths = [
        KAGGLE_ABLATION_DIR / "leaderboard_live.csv",
        KAGGLE_ABLATION_DIR / "leaderboard_live2.csv",
    ]
    existing_paths = [path for path in leaderboard_paths if path.exists()]
    if existing_paths:
        leaderboard_df = pd.concat([pd.read_csv(path) for path in existing_paths], ignore_index=True)
        leaderboard_df = leaderboard_df.sort_values("f1", ascending=False).drop_duplicates(subset=["run_name"])
        leaderboard_df = leaderboard_df.query("teacher_layer == 'layer1'").sort_values("f1", ascending=False)
        for _, row in leaderboard_df.iterrows():
            analysis_rows.append(
                {
                    "variant": row["run_name"],
                    "source": "kaggle_layer1",
                    "teacher_layer": row["teacher_layer"],
                    "topk_ratio": float(row["topk_ratio"]),
                    "score_student_weight": float(row["score_student_weight"]),
                    "score_autoencoder_weight": float(row["score_auto_weight"]),
                    "precision": float(row["precision"]),
                    "recall": float(row["recall"]),
                    "f1": float(row["f1"]),
                    "auroc": float(row["auroc"]),
                    "auprc": float(row["auprc"]),
                    "best_sweep_f1": float(row["best_sweep_f1"]),
                    "threshold": float(row["threshold"]),
                    "best_epoch": int(row["best_epoch"]),
                    "best_val_loss": float(row["best_val_loss"]),
                    "output_dir": str(row["run_dir"]),
                    "config_path": "kaggle_inline_notebook",
                }
            )
    else:
        print("No Kaggle leaderboard CSVs found. Skipping saved layer1 rows.")

analysis_df = pd.DataFrame(analysis_rows).sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
analysis_df.head(10).round(6)

,variant,source,teacher_layer,topk_ratio,score_student_weight,score_autoencoder_weight,precision,recall,f1,auroc,auprc,best_sweep_f1,threshold,best_epoch,best_val_loss,output_dir,config_path
0,tsres50_layer1_topk0p1_sw1p0_aw0p5,kaggle_layer1,layer1,0.10,1.0,0.5,0.407862,0.664,0.505327,0.872754,0.527526,0.547284,4.805169,30.0,0.417623,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
1,tsres50_layer1_topk0p1_sw2p0_aw1p0,kaggle_layer1,layer1,0.10,2.0,1.0,0.401460,0.660,0.499244,0.872678,0.521696,0.545906,9.634031,26.0,0.418919,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
2,tsres50_layer1_topk0p1_sw2p0_aw0p5,kaggle_layer1,layer1,0.10,2.0,0.5,0.396226,0.672,0.498516,0.867110,0.518197,0.564516,7.781686,28.0,0.417378,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
3,TS-Res50 selected local score,imported_layer2,layer2,0.20,2.0,1.0,0.382353,0.676,0.488439,0.912730,0.581275,0.560461,2.257497,NaN,NaN,C:\Users\User\Desktop\Term 8\Deep Learning\Pro...,C:\Users\User\Desktop\Term 8\Deep Learning\Pro...
4,TS-Res50 imported default,imported_layer2,layer2,0.20,1.0,0.0,0.382353,0.676,0.488439,0.912691,0.581770,0.562141,2.255374,NaN,NaN,C:\Users\User\Desktop\Term 8\Deep Learning\Pro...,C:\Users\User\Desktop\Term 8\Deep Learning\Pro...
5,tsres50_layer1_topk0p2_sw2p0_aw1p0,kaggle_layer1,layer1,0.20,2.0,1.0,0.399491,0.628,0.488336,0.863550,0.453591,0.535865,8.410694,29.0,0.415842,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
6,tsres50_layer1_topk0p2_sw1p0_aw0p5,kaggle_layer1,layer1,0.20,1.0,0.5,0.393939,0.624,0.482972,0.863149,0.449665,0.528067,4.206496,30.0,0.416230,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
7,tsres50_layer1_topk0p15_sw1p0_aw0p5,kaggle_layer1,layer1,0.15,1.0,0.5,0.388206,0.632,0.480974,0.868794,0.482786,0.539370,4.454726,29.0,0.412657,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
8,tsres50_layer1_topk0p1_sw1p0_aw1p0,kaggle_layer1,layer1,0.10,1.0,1.0,0.381643,0.632,0.475904,0.874714,0.529302,0.541284,6.806980,23.0,0.416822,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook
9,tsres50_layer1_topk0p2_sw1p0_aw1p0,kaggle_layer1,layer1,0.20,1.0,1.0,0.379052,0.608,0.466974,0.868630,0.455612,0.513131,5.862299,30.0,0.415980,/kaggle/working/ts_resnet50_focused_ablation/r...,kaggle_inline_notebook


## Optional Local Rerun

Set `RUN_VARIANT_SWEEP = True` in the setup cell above if you want to train and evaluate local `TS-ResNet50` variants from this repo.

The notebook uses the same repo scripts as the other experiment notebooks:
- `scripts/train_ts_distillation.py`
- `scripts/evaluate_reconstruction_model.py`

That keeps the workflow reproducible, while still letting us reuse existing artifacts when a full rerun would be too expensive.

In [ ]:
if RUN_VARIANT_SWEEP:
    local_rows = []
    for spec in VARIANT_SPECS:
        print(f"\n=== Variant: {spec['name']} ===")
        local_rows.append(
            run_variant(
                spec=spec,
                threshold_quantile=THRESHOLD_QUANTILE,
                reuse_existing=REUSE_EXISTING_LOCAL_RUNS,
            )
        )
    local_df = pd.DataFrame(local_rows).sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
    local_df.to_csv(LOCAL_SUMMARY_PATH, index=False)
    print(f"Saved local summary to {LOCAL_SUMMARY_PATH}")
elif LOCAL_SUMMARY_PATH.exists():
    local_df = pd.read_csv(LOCAL_SUMMARY_PATH)
    print(f"Loaded existing local summary from {LOCAL_SUMMARY_PATH}")
else:
    local_df = pd.DataFrame()
    print("Skipping local rerun. Set RUN_VARIANT_SWEEP = True to launch it.")

local_df.round(6)

In [ ]:
comparison_frames = [df for df in [analysis_df, local_df] if not df.empty]
if comparison_frames:
    comparison_df = pd.concat(comparison_frames, ignore_index=True)
    comparison_df = comparison_df.sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
    display(comparison_df[[
        "variant",
        "source",
        "teacher_layer",
        "topk_ratio",
        "score_student_weight",
        "score_autoencoder_weight",
        "precision",
        "recall",
        "f1",
        "auroc",
        "auprc",
        "best_sweep_f1",
    ]].round(6))
else:
    comparison_df = pd.DataFrame()
    print("No comparison rows available yet.")

comparison_df

## Notes

- `layer2` remains the strongest verified `TS-Res50` direction currently saved in the repo.
- `layer1` is still a real experimental branch, not just a one-off artifact, and this notebook makes that explicit.
- The default notebook behavior is safe and lightweight: it analyzes saved artifacts first.
- When hardware is available, switching on `RUN_VARIANT_SWEEP` makes the notebook runnable end-to-end from this repo.